# Activation Functions: From Sigmoid to GELU

[← Back to wiki](https://ml-viz-ruby.vercel.app/wiki/activation-functions)

We plot every common activation and its derivative, demonstrate the dead-ReLU problem, trace how saturation causes vanishing gradients through depth, and implement numerically stable softmax.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
plt.style.use('dark_background')
rng = np.random.default_rng(0)

## 1 — The activation zoo and their derivatives

In [ ]:
sigmoid = lambda z: 1/(1+np.exp(-z))
tanh    = lambda z: np.tanh(z)
relu    = lambda z: np.maximum(0, z)
lrelu   = lambda z, a=0.1: np.where(z>0, z, a*z)
elu     = lambda z, a=1.0: np.where(z>0, z, a*(np.exp(z)-1))
gelu    = lambda z: z*norm.cdf(z)
silu    = lambda z: z*sigmoid(z)   # Swish

# derivatives
d_sigmoid = lambda z: sigmoid(z)*(1-sigmoid(z))
d_tanh    = lambda z: 1-np.tanh(z)**2
d_relu    = lambda z: (z>0).astype(float)

z = np.linspace(-5, 5, 400)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for fn, name in [(sigmoid,'sigmoid'),(tanh,'tanh'),(relu,'relu'),(lrelu,'leaky'),(gelu,'gelu'),(silu,'silu')]:
    ax[0].plot(z, fn(z), label=name)
ax[0].set_title('Activations'); ax[0].legend(fontsize=8); ax[0].axhline(0,color='gray',lw=0.5); ax[0].axvline(0,color='gray',lw=0.5)
for fn, name in [(d_sigmoid,"sigmoid'"),(d_tanh,"tanh'"),(d_relu,"relu'")]:
    ax[1].plot(z, fn(z), label=name)
ax[1].axhline(0.25, color='#f87171', ls='--', label='sigmoid max=0.25')
ax[1].set_title('Derivatives (note sigmoid ceiling)'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2 — The dead ReLU problem

A neuron pushed negative for every input outputs 0 with gradient 0 — and can never recover.

In [ ]:
X = rng.normal(0, 1, (1000, 4))
# A unit with a large negative bias: pre-activation is negative for ~all inputs
w = rng.normal(0, 0.5, 4)
b_dead = -6.0
z_dead = X @ w + b_dead
print(f"ReLU unit with bias {b_dead}: active on {(z_dead>0).mean()*100:.1f}% of inputs")
print(f"  -> gradient is 0 on {(z_dead<=0).mean()*100:.1f}% of inputs (this unit is effectively dead)")

# Leaky ReLU keeps a small gradient alive
grad_relu  = d_relu(z_dead).mean()
grad_leaky = np.where(z_dead>0, 1.0, 0.1).mean()
print(f"\nMean gradient   ReLU: {grad_relu:.3f}   LeakyReLU: {grad_leaky:.3f}")
print("LeakyReLU still passes gradient, so the unit can recover.")

## 3 — Saturation -> vanishing gradients through depth

Backprop multiplies activation derivatives layer by layer. Sigmoid's 0.25 ceiling shrinks the signal geometrically; ReLU's 1.0 preserves it.

In [ ]:
depths = np.arange(1, 21)
# Best case: every derivative at its maximum
sig_grad  = 0.25 ** depths
relu_grad = 1.0  ** depths

plt.figure(figsize=(8, 4))
plt.semilogy(depths, sig_grad, 'o-', label='sigmoid (0.25^depth)', color='#f87171')
plt.semilogy(depths, relu_grad, 's-', label='relu (1.0^depth)', color='#34d399')
plt.xlabel('depth (layers)'); plt.ylabel('gradient scale (log)')
plt.title('Why saturating activations kill deep nets'); plt.legend()
plt.tight_layout(); plt.show()
print(f"At depth 10: sigmoid scales gradient by {0.25**10:.2e}, relu by {1.0**10:.0f}")

## 4 — Numerically stable softmax

Softmax is shift-invariant, so subtract the max logit before exponentiating to avoid overflow.

In [ ]:
def softmax_naive(z):
    e = np.exp(z); return e / e.sum()

def softmax_stable(z):
    e = np.exp(z - z.max()); return e / e.sum()

logits = np.array([1000.0, 1001.0, 1002.0])  # huge logits
print("naive: ", softmax_naive(logits), "<- overflow to nan")
print("stable:", softmax_stable(logits).round(4), "<- correct")
assert np.isclose(softmax_stable(logits).sum(), 1.0)

## ✏️ Your turn

**Task A — tanh approximation of GELU:** Implement the popular tanh approximation $0.5z(1 + \tanh[\sqrt{2/\pi}(z + 0.044715 z^3)])$ and show its max absolute error vs the exact $z\,\Phi(z)$ over $z \in [-5,5]$.

**Task B — empirical dead-unit rate:** For a layer of 256 ReLU units with He-initialized weights on standard-normal input, measure what fraction of units are dead (zero output on the whole batch). Repeat with a too-large negative bias and show the rate spikes.

In [ ]:
def gelu_tanh(z):
    # TODO(you): implement the tanh approximation of GELU
    return ...

z = np.linspace(-5, 5, 1000)
approx = gelu_tanh(z)
if approx is not None:
    err = np.abs(approx - gelu(z)).max()
    print(f"Max abs error of tanh-GELU vs exact: {err:.6f}")

<details><summary>Solution — Task A</summary>

```python
def gelu_tanh(z):
    return 0.5*z*(1 + np.tanh(np.sqrt(2/np.pi)*(z + 0.044715*z**3)))
# Max error is ~0.0003 — why the approximation is used interchangeably in practice.
```
</details>